# 00 - Data Preparation
**SuperCognition Labs - synthetic handwriting for CAPTCHA training**

Goal of this notebook: download MNIST, apply the standard transform (`[0,255] -> [0,1] -> [-1,1]`), check shapes / value range / class balance and look at a few real digits.
Both generative models in this project work in the `[-1, 1]` range (the cGAN generator ends in `Tanh`, and the diffusion model adds Gaussian noise around zero).

In [ ]:
import os, sys, random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Set QUICK_RUN=1 in the environment for a tiny smoke-test run (few samples / 1 epoch). Leave unset for the real run.
QUICK_RUN = os.environ.get("QUICK_RUN", "0") == "1"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| QUICK_RUN:", QUICK_RUN)
os.makedirs("checkpoints", exist_ok=True); os.makedirs("outputs", exist_ok=True)

In [ ]:
from data_utils.dataloader import get_mnist_dataloaders, get_mnist_dataset, dataset_to_tensors
from utils.visualize import show_images, show_class_grid

train_loader, test_loader = get_mnist_dataloaders(batch_size=128)
print("train batches:", len(train_loader), "| test batches:", len(test_loader))
print("train samples:", len(train_loader.dataset), "| test samples:", len(test_loader.dataset))

### Inspect one batch (shape, dtype, value range)

In [ ]:
images, labels = next(iter(train_loader))
print("images:", tuple(images.shape), images.dtype, "| labels:", tuple(labels.shape), labels.dtype)
print(f"value range: [{images.min():.2f}, {images.max():.2f}]  (expected [-1, 1])")
assert images.shape[1:] == (1, 28, 28) and images.min() >= -1 and images.max() <= 1

### Class balance

In [ ]:
train_x, train_y = dataset_to_tensors(train=True)
test_x, test_y = dataset_to_tensors(train=False)
counts = torch.bincount(train_y, minlength=10)
plt.figure(figsize=(6, 3)); plt.bar(range(10), counts.numpy()); plt.xticks(range(10))
plt.xlabel("digit"); plt.ylabel("# training images"); plt.title("MNIST training set class distribution"); plt.show()
print("train:", tuple(train_x.shape), "test:", tuple(test_x.shape))

### Visualise real digits

In [ ]:
show_images(images[:64], nrow=8, title="Random real MNIST training digits", save_path="outputs/00_real_samples.png")
show_class_grid(train_x, train_y, per_class=10, title="Real MNIST - one row per class", save_path="outputs/00_real_class_grid.png")

### Summary
* MNIST is loaded through `data/dataloader.py`, normalised to `[-1, 1]`, shape `(1, 28, 28)`.
* Classes are roughly balanced, so uniformly sampled labels during generation are appropriate.
* Next: `01_cGAN_training.ipynb` and `02_diffusion_training.ipynb`.